# Nodes, Edges, and Graph Compilation

**01. Introduction to LangGraph** ran a minimal graph; **02. StateGraph and Shared State** defined schemas and reducers. This notebook focuses on **wiring**: registering **nodes**, connecting them with **edges**, and **`compile()`** — the step that produces a runnable **`CompiledStateGraph`** with `invoke`, `stream`, and checkpoint support.

Conditional routing (`add_conditional_edges`) is **04. Conditional Routing and Branching** — here we master fixed edges, `START`/`END`, Runnable nodes, and compile options.

Prerequisites: **01. Introduction to LangGraph**, **02. StateGraph and Shared State**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

import langchain_core

try:
    import langgraph
    print("langgraph:", langgraph.__version__)
except ImportError:
    print("langgraph: pip install langgraph")

print("langchain-core:", langchain_core.__version__)

---

## 1. Builder vs Compiled Graph

LangGraph development is two phases:

```
Phase 1 — BUILD (mutable)          Phase 2 — COMPILE (frozen)
─────────────────────────          ───────────────────────────
builder = StateGraph(State)   →    graph = builder.compile(...)
builder.add_node(...)              graph.invoke(...)
builder.add_edge(...)              graph.stream(...)
                                     graph.get_state(...)  # with checkpointer
```

| Object | Type | Runnable? |
|--------|------|----------|
| **`StateGraph`** | Builder | No — must `compile()` |
| **`CompiledStateGraph`** | Executable app | Yes — `invoke`, `batch`, `astream` |

You **cannot** invoke a builder directly — always call **`compile()`** first.

---

## 2. State Schema Recap

Every graph is bound to a state type from **02**:

In [ ]:
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph


class PipelineState(TypedDict):
    question: str
    context: str
    answer: str
    trace: list[str]

---

## 3. Nodes — Functions That Patch State

A **node** is a callable with signature **`(state) -> dict`** (optional second `config` arg). It **reads** state and returns a **partial update**:

In [ ]:
DOCS = {
    "alembic": "Run alembic upgrade head after pulling migrations.",
    "jwt": "JWT bearer tokens use Authorization: Bearer header.",
    "pytest": "Pytest DB fixtures belong in conftest.py.",
}


def load_context(state: PipelineState) -> dict:
    q = state["question"].lower()
    for key, text in DOCS.items():
        if key in q:
            return {"context": f"[{key}] {text}", "trace": state.get("trace", []) + ["load_context"]}
    return {"context": "(no match)", "trace": state.get("trace", []) + ["load_context:empty"]}


def generate_answer(state: PipelineState) -> dict:
    answer = f"Answer for '{state['question']}': {state['context']}"
    return {"answer": answer, "trace": state.get("trace", []) + ["generate_answer"]}


def finalize(state: PipelineState) -> dict:
    return {"trace": state.get("trace", []) + ["finalize"]}

### 3.1 Node Naming

| Convention | Example | Why |
|------------|---------|-----|
| **snake_case** | `load_context` | Matches Python style |
| **Verb-first** | `retrieve_docs`, `call_model` | Describes action |
| **Stable names** | Don't rename in prod | Breaks checkpoints and traces |

Node names appear in **LangSmith traces**, **stream events**, and **interrupt** configs.

---

## 4. Registering Nodes — add_node

**`builder.add_node(name, callable)`** registers a step. The name is the edge target label:

In [ ]:
builder = StateGraph(PipelineState)

builder.add_node("load_context", load_context)
builder.add_node("generate_answer", generate_answer)
builder.add_node("finalize", finalize)

print("registered nodes (builder):", builder.nodes.keys() if hasattr(builder, "nodes") else "load_context, generate_answer, finalize")

You can register the **same function** under different names if you need two instances with different config — rare; prefer separate functions for clarity.

---

## 5. START and END

LangGraph provides sentinel nodes **`START`** and **`END`**:

In [ ]:
print("START:", START)
print("END:", END)

| Constant | Role |
|----------|------|
| **`START`** | Virtual entry — first edge must leave `START` |
| **`END`** | Virtual exit — graph stops when a node reaches `END` |

Every executable path must flow **`START → … → END`**. Orphan nodes (no path from `START`) never run.

---

## 6. Fixed Edges — add_edge

**`add_edge(from, to)`** always transitions `from → to`:

In [ ]:
builder.add_edge(START, "load_context")
builder.add_edge("load_context", "generate_answer")
builder.add_edge("generate_answer", "finalize")
builder.add_edge("finalize", END)

print("edges wired: START → load_context → generate_answer → finalize → END")

```
START ──► load_context ──► generate_answer ──► finalize ──► END
```

This is a **linear DAG** (no cycles). Agent loops add edges back from `tools` to `model` (**06**).

---

## 7. compile() — Freeze the Graph

**`compile()`** validates wiring and returns a **`CompiledStateGraph`**:

In [ ]:
graph = builder.compile()

result = graph.invoke({
    "question": "What pytest file holds fixtures?",
    "context": "",
    "answer": "",
    "trace": [],
})

print(json.dumps(result, indent=2))

At compile time LangGraph checks that registered nodes are reachable and edge targets exist. Miswired graphs fail **before** production traffic hits them.

---

## 8. Inspecting the Compiled Graph

Use **`get_graph()`** for structure review and documentation:

In [ ]:
g = graph.get_graph()
print("nodes:", list(g.nodes))
print("edges:", list(g.edges))

In [ ]:
try:
    mermaid = g.draw_mermaid()
    print(mermaid[:300], "...")
except Exception as exc:
    print("draw_mermaid:", type(exc).__name__, "— optional dependency may be missing")

Paste Mermaid output into docs or PRs so reviewers see control flow without reading Python.

---

## 9. Nodes as LangChain Runnables

Wrap LCEL chains (**01. LangChain/02**) inside node functions, or pass a **Runnable** when the input/output maps cleanly to state:

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY this context.\n{context}"),
    ("human", "{question}"),
])
qa_chain = qa_prompt | llm | StrOutputParser()


def run_qa_chain(state: PipelineState) -> dict:
    answer = qa_chain.invoke({"context": state["context"], "question": state["question"]})
    return {"answer": answer, "trace": state.get("trace", []) + ["run_qa_chain"]}


# RunnableLambda wrapper pattern for simple transforms
def to_answer_dict(text: str) -> dict:
    return {"answer": text}


echo_runnable = RunnableLambda(lambda s: f"Echo: {s['question']}") | RunnableLambda(to_answer_dict)


def runnable_node(state: PipelineState) -> dict:
    """Bridge state dict to Runnable — pattern only."""
    patch = echo_runnable.invoke(state)
    patch["trace"] = state.get("trace", []) + ["runnable_node"]
    return patch

Production pattern: **one LCEL chain per node function** — keeps state mapping explicit and testable.

---

## 10. Config-Aware Nodes

Nodes may accept **`config`** as a second argument for tags, metadata, and `configurable` fields (**01. LangChain/16**):

In [ ]:
def config_aware_node(state: PipelineState, config) -> dict:
    tier = config.get("configurable", {}).get("user_tier", "free")
    prefix = "[PRO]" if tier == "pro" else "[FREE]"
    return {
        "answer": f"{prefix} {state.get('answer', '')}",
        "trace": state.get("trace", []) + [f"config_aware:{tier}"],
    }


cfg_builder = StateGraph(PipelineState)
cfg_builder.add_node("load", load_context)
cfg_builder.add_node("gen", generate_answer)
cfg_builder.add_node("tag", config_aware_node)
cfg_builder.add_edge(START, "load")
cfg_builder.add_edge("load", "gen")
cfg_builder.add_edge("gen", "tag")
cfg_builder.add_edge("tag", END)

cfg_graph = cfg_builder.compile()

pro_out = cfg_graph.invoke(
    {"question": "jwt header", "context": "", "answer": "", "trace": []},
    config={"configurable": {"user_tier": "pro"}},
)
print(pro_out["answer"])

`config` propagates to nested Runnables invoked inside the node when you pass it through.

---

## 11. compile() Options

Pass runtime behavior at compilation — not on every `invoke`:

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

production_graph = builder.compile(
    checkpointer=checkpointer,
    interrupt_before=[],   # e.g. ["execute_side_effect"] — see 09
    interrupt_after=[],
    debug=False,
)

thread_cfg = {"configurable": {"thread_id": "compile-demo-1"}}
r1 = production_graph.invoke(
    {"question": "Alembic command?", "context": "", "answer": "", "trace": []},
    config=thread_cfg,
)
print("checkpointed answer:", r1["answer"][:80], "...")

| `compile()` argument | Purpose |
|---------------------|--------|
| **`checkpointer`** | Persist state per `thread_id` (**08**) |
| **`interrupt_before`** | Pause before listed nodes (HITL) (**09**) |
| **`interrupt_after`** | Pause after listed nodes |
| **`debug`** | Verbose logging during development |

Recompile when checkpointer or interrupt policy changes — not per request.

---

## 12. CompiledStateGraph — Runnable API

Compiled graphs implement the LangChain **Runnable** protocol (**01. LangChain/02**):

In [ ]:
print("invoke:  ", hasattr(graph, "invoke"))
print("stream:  ", hasattr(graph, "stream"))
print("batch:   ", hasattr(graph, "batch"))
print("ainvoke: ", hasattr(graph, "ainvoke"))
print("astream: ", hasattr(graph, "astream"))

In [ ]:
print("Streaming trace steps:")
for step in graph.stream(
    {"question": "jwt auth", "context": "", "answer": "", "trace": []},
    stream_mode="updates",
):
    print(step)

**`stream_mode="updates"`** yields per-node patches — ideal for progress UI. Full streaming coverage: **13. Streaming, Batch, and Async**.

---

## 13. Graph Construction Checklist

Use this workflow for every new graph:

1. **Design state schema** — keys, reducers (**02**)
2. **List nodes** — one responsibility each
3. **Sketch ASCII diagram** — include `START`/`END`
4. **`StateGraph(schema)`** — create builder
5. **`add_node`** for each step
6. **`add_edge`** for fixed transitions
7. **`add_conditional_edges`** where routing is dynamic (**04**)
8. **`compile(...)`** with checkpointer if multi-turn
9. **`get_graph().draw_mermaid()`** — attach to design doc
10. **Smoke `invoke`** + stream updates — verify trace order

---

## 14. Fixed Edges vs Conditional (Preview)

| Mechanism | When |
|-----------|------|
| **`add_edge(a, b)`** | Always go `a → b` |
| **`add_conditional_edges`** | Router function picks next node |

```python
# Preview only — full treatment in 04
builder.add_conditional_edges("grade", route_fn, {"relevant": "generate", "irrelevant": "rewrite"})
```

Agent tool loops use conditional edges from **`model`** to **`tools`** or **`END`** (**06**, **07**).

---

## 15. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Invoking builder without `compile()` | AttributeError | `graph = builder.compile()` |
| No path from `START` | Node never runs | Trace edges from `START` |
| Edge to unknown node name | Compile error | Match `add_node` names exactly |
| Dead-end node (no edge to `END`) | Hang or error | Terminate all paths at `END` |
| Business logic in edges | Untestable routing | Logic in nodes or router fns (**04**) |
| Rebuilding graph per request | Slow, leaky | Build once at app startup (**16**) |
| Renaming nodes after launch | Broken checkpoints | Treat names as API |

---

## 16. Summary

**Nodes** are named callables that patch state; **edges** connect them; **`START`** and **`END`** bound execution. **`compile()`** validates the builder and returns a **`CompiledStateGraph`** — a Runnable with `invoke`, `stream`, checkpointing, and optional interrupts.

Key takeaways:

- **Builder** = mutable wiring; **compiled graph** = executable.
- **`add_node(name, fn)`** + **`add_edge(from, to)`** define fixed flows.
- Wrap **LCEL chains** inside node functions for clarity.
- Nodes may read **`config`** for per-request tier and tracing.
- **`compile(checkpointer=..., interrupt_before=...)`** sets production behavior.
- **`get_graph()`** / Mermaid export documents control flow.
- **Conditional edges** come next (**04**); cycles in **06**.

Demonstrations registered a linear three-node pipeline, compiled and invoked it, inspected graph structure, wrapped LCEL in nodes, used config-aware nodes, applied compile options with a checkpointer, and streamed per-node updates.

Next: **04. Conditional Routing and Branching** — `add_conditional_edges`, routers, and dynamic paths.